# Práctica 6. Retrieval bajo medición

Starter de la práctica. Sigue `practice.md` y llena `report-template.md` con las tablas que imprime este notebook. Corre en Colab sin llave: todo es retrieval con modelos de embeddings locales.

Las celdas marcadas `TODO` tienen placeholders para que el notebook corra de principio a fin desde el inicio; sustitúyelos conforme avances.

In [ ]:
!pip install -q sentence-transformers openai 2>/dev/null || true
import os, re, json, time, warnings
warnings.filterwarnings("ignore"); os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"; os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np, pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)

In [ ]:
# Llave de OpenAI. En Colab: panel de secretos (icono de llave) con el nombre OPENAI_API_KEY
# y acceso activado para este notebook. En Jupyter local: variable de entorno antes de abrir.
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # fuera de Colab, o sin secreto: se usa la variable de entorno si existe
print("llave cargada" if os.environ.get("OPENAI_API_KEY") else "sin llave: se usará el modelo local")

In [ ]:
# Copia de examples/rag_lib.py para que el starter corra en Colab sin clonar el repositorio.
# La versión canónica de las métricas está en course/shared/retrieval_eval.py.
from __future__ import annotations

import json
import re
import urllib.request
from pathlib import Path

import numpy as np

ARCHIVOS = [
    "01-manual-facturacion-cfdi.md", "02-manual-usuarios-e-invitaciones.md",
    "03-manual-reportes-y-exportacion.md", "04-manual-integraciones-y-api.md",
    "05-preguntas-frecuentes.md", "06-politica-reembolsos-y-cargos.md",
    "07-politica-cancelacion-y-eliminacion.md", "08-terminos-del-servicio-y-sla.md",
    "09-catalogo-codigos-de-error.md", "10-notas-de-version.md",
    "11-guia-de-seguridad.md", "12-comunicado-externo.md",
]
TIPO_DOC = {"01": "manual", "02": "manual", "03": "manual", "04": "manual", "05": "faq",
            "06": "politica", "07": "politica", "08": "terminos", "09": "catalogo",
            "10": "notas", "11": "seguridad", "12": "comunicado"}
RAW = "https://raw.githubusercontent.com/JunBarrs24/itm-maestria-llms/main/shared/datasets/corpus/"


# ------------------------------------------------------------------ carga
def cargar_corpus(local: str | Path | None = None) -> dict[str, str]:
    """Devuelve {archivo: texto}. Usa la carpeta local si existe; si no, descarga del repositorio."""
    docs = {}
    base = Path(local) if local else None
    for nombre in ARCHIVOS:
        if base and (base / nombre).exists():
            docs[nombre] = (base / nombre).read_text(encoding="utf-8")
        else:
            with urllib.request.urlopen(RAW + nombre) as r:
                docs[nombre] = r.read().decode("utf-8")
    return docs


def cargar_gold(local: str | Path | None = None) -> list[dict]:
    base = Path(local) if local else None
    if base and (base / "gold_set.jsonl").exists():
        texto = (base / "gold_set.jsonl").read_text(encoding="utf-8")
    else:
        with urllib.request.urlopen(RAW + "gold_set.jsonl") as r:
            texto = r.read().decode("utf-8")
    return [json.loads(l) for l in texto.splitlines() if l.strip()]


# ------------------------------------------------------------------ estructura
def secciones(texto: str) -> list[dict]:
    """Parte un markdown por encabezados ## y ###. Cada sección conserva su encabezado."""
    out, actual, buf = [], None, []
    for linea in texto.splitlines():
        m = re.match(r"^(#{2,3})\s+(.*)", linea)
        if m:
            if actual is not None:
                out.append({"seccion": actual, "texto": "\n".join(buf).strip()})
            actual, buf = m.group(2).strip(), [linea]
        else:
            buf.append(linea)
    if actual is not None:
        out.append({"seccion": actual, "texto": "\n".join(buf).strip()})
    return out


def seccion_en(texto: str, pos: int) -> str | None:
    """Encabezado ## o ### vigente en la posición pos del texto."""
    ultimo = None
    for m in re.finditer(r"^(#{2,3})\s+(.*)$", texto[:pos + 1], re.M):
        ultimo = m.group(2).strip()
    return ultimo


# ------------------------------------------------------------------ chunking
def chunk_por_seccion(docs: dict[str, str]) -> list[dict]:
    chunks = []
    for archivo, texto in docs.items():
        for s in secciones(texto):
            chunks.append({"archivo": archivo, "seccion": s["seccion"], "tipo": TIPO_DOC[archivo[:2]],
                           "texto": s["texto"]})
    return chunks


def chunk_seccion_contexto(docs: dict[str, str], minimo_palabras: int = 8) -> list[dict]:
    """Como chunk_por_seccion, pero cada chunk lleva al inicio su ruta jerárquica
    (título del documento > sección ## > subsección ###) y se omiten los encabezados
    sin cuerpo propio. El texto extra le da al embedding el contexto que el fragmento perdió."""
    out = []
    for archivo, texto in docs.items():
        m = re.search(r"^# (.+)$", texto, re.M)
        titulo = m.group(1).strip() if m else archivo
        padre = None
        for s in secciones(texto):
            nivel2 = s["texto"].startswith("## ")
            cuerpo = "\n".join(s["texto"].splitlines()[1:]).strip()
            if nivel2:
                padre = s["seccion"]
            if len(cuerpo.split()) < minimo_palabras:
                continue
            ruta = f"{titulo} > {padre} > {s['seccion']}" if (not nivel2 and padre) else f"{titulo} > {s['seccion']}"
            out.append({"archivo": archivo, "seccion": s["seccion"], "tipo": TIPO_DOC[archivo[:2]],
                        "texto": ruta + "\n\n" + cuerpo})
    return out


def chunk_fijo(docs: dict[str, str], tamano: int, overlap: int, contar) -> list[dict]:
    """Ventanas de `tamano` tokens con `overlap` tokens de traslape, medidas en palabras.
    `contar(texto)` devuelve tokens; se usa para calibrar palabras por token en cada documento.
    Un chunk fijo puede cruzar varias secciones: `seccion` es la vigente al inicio y
    `secciones` la lista de todas las que toca."""
    chunks = []
    for archivo, texto in docs.items():
        palabras = [(m.group(0), m.start(), m.end()) for m in re.finditer(r"\S+", texto)]
        if not palabras:
            continue
        encabezados = [(m.start(), m.group(2).strip()) for m in re.finditer(r"^(#{2,3})\s+(.*)$", texto, re.M)]
        ratio = contar(texto) / len(palabras)              # tokens por palabra en este documento
        paso_p = max(1, int(tamano / ratio))                # palabras por chunk
        over_p = int(overlap / ratio)
        i = 0
        while i < len(palabras):
            ventana = palabras[i:i + paso_p]
            ini, fin = ventana[0][1], ventana[-1][2]
            vigente = None
            tocadas = []
            for pos, h in encabezados:
                if pos <= ini:
                    vigente = h
                elif pos < fin:
                    tocadas.append(h)
            secs = ([vigente] if vigente else []) + tocadas
            chunks.append({"archivo": archivo, "seccion": vigente, "secciones": secs,
                           "tipo": TIPO_DOC[archivo[:2]], "texto": texto[ini:fin]})
            if i + paso_p >= len(palabras):
                break
            i += max(1, paso_p - over_p)
    return chunks


def chunk_recursivo(docs: dict[str, str], tamano: int, contar) -> list[dict]:
    """Parte por secciones; si una sección excede `tamano` tokens, la parte por párrafos;
    si un párrafo excede, por oraciones. Nunca corta a media palabra."""
    def partir(texto: str) -> list[str]:
        if contar(texto) <= tamano:
            return [texto]
        partes = [p for p in re.split(r"\n\s*\n", texto) if p.strip()]
        if len(partes) == 1:
            partes = [p for p in re.split(r"(?<=[.!?])\s+", texto) if p.strip()]
            if len(partes) == 1:
                return [texto]
        out, buf = [], ""
        for p in partes:
            cand = (buf + "\n\n" + p).strip() if buf else p
            if contar(cand) <= tamano:
                buf = cand
            else:
                if buf:
                    out.append(buf)
                buf = p
        if buf:
            out.append(buf)
        return out
    chunks = []
    for archivo, texto in docs.items():
        for s in secciones(texto):
            for t in partir(s["texto"]):
                chunks.append({"archivo": archivo, "seccion": s["seccion"], "tipo": TIPO_DOC[archivo[:2]], "texto": t})
    return chunks


# ------------------------------------------------------------------ índice y retrieval
class Indice:
    """Índice en numpy: embeddings normalizados, similitud coseno por producto punto."""

    def __init__(self, modelo, chunks: list[dict], max_seq_length: int | None = None):
        self.modelo, self.chunks = modelo, chunks
        if max_seq_length:
            modelo.max_seq_length = max_seq_length
        self.E = modelo.encode([c["texto"] for c in chunks], normalize_embeddings=True, batch_size=32,
                               show_progress_bar=False)

    def buscar(self, pregunta: str, k: int = 5, filtro=None) -> list[dict]:
        q = self.modelo.encode([pregunta], normalize_embeddings=True, show_progress_bar=False)[0]
        scores = self.E @ q
        orden = np.argsort(-scores)
        out = []
        for i in orden:
            c = self.chunks[i]
            if filtro and not filtro(c):
                continue
            out.append({**c, "score": float(scores[i])})
            if len(out) == k:
                break
        return out


# ------------------------------------------------------------------ métricas (copia de shared/retrieval_eval.py)
def es_relevante(chunk: dict, evidencias: list[dict], por_seccion: bool = True) -> bool:
    for ev in evidencias:
        if chunk.get("archivo") != ev.get("archivo"):
            continue
        if not por_seccion or not ev.get("seccion"):
            return True
        if (chunk.get("seccion") or "") == ev["seccion"]:
            return True
        if ev["seccion"] in (chunk.get("secciones") or []):   # chunk fijo que cruza secciones
            return True
    return False


def metricas_pregunta(chunks: list[dict], evidencias: list[dict], k: int, por_seccion: bool = True) -> dict:
    top = chunks[:k]
    hits = [es_relevante(c, evidencias, por_seccion) for c in top]
    encontradas = set()
    for c in top:
        for ev in evidencias:
            if es_relevante(c, [ev], por_seccion):
                encontradas.add(ev["archivo"] + "#" + (ev.get("seccion") or "" if por_seccion else ""))
    total = len({ev["archivo"] + "#" + (ev.get("seccion") or "" if por_seccion else "") for ev in evidencias})
    mrr = next((1.0 / i for i, h in enumerate(hits, start=1) if h), 0.0)
    return {"hit": float(any(hits)), "recall": len(encontradas) / total if total else 0.0, "mrr": mrr}


def evaluar(gold: list[dict], retriever, k: int = 5, por_seccion: bool = True, tipos: set | None = None) -> dict:
    filas = []
    for g in gold:
        if not g.get("evidencia"):
            continue
        if tipos and g.get("tipo") not in tipos:
            continue
        m = metricas_pregunta(retriever(g["pregunta"]), g["evidencia"], k, por_seccion)
        filas.append({**m, "id": g["id"], "tipo": g["tipo"]})
    n = len(filas)
    return {"k": k, "preguntas": n,
            f"hit@{k}": round(sum(f["hit"] for f in filas) / n, 3) if n else 0.0,
            f"recall@{k}": round(sum(f["recall"] for f in filas) / n, 3) if n else 0.0,
            "mrr": round(sum(f["mrr"] for f in filas) / n, 3) if n else 0.0,
            "por_pregunta": filas}

In [ ]:
LOCAL = Path("../../shared/datasets/corpus")
docs = cargar_corpus(LOCAL if LOCAL.exists() else None)
gold = cargar_gold(LOCAL if LOCAL.exists() else None)
MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
modelo = SentenceTransformer(MODELO); modelo.max_seq_length = 512
contar = lambda t: len(modelo.tokenizer.encode(t, add_special_tokens=False))
print(len(docs), "documentos |", len(gold), "preguntas en el gold set | dim", modelo.get_sentence_embedding_dimension(), "| max_seq_length", modelo.max_seq_length)

## Sección 1. Predicciones y barrido

In [ ]:
# TODO: escribe las tres predicciones ANTES de ejecutar el barrido. Se conservan aunque fallen.
PREDICCIONES = {
    "hit@3 con chunks de 500 frente a 200": None,          # "sube" | "baja" | "igual" + por qué
    "estrategia con más tokens de contexto @5": None,      # "fijo 200" | "fijo 1000" | "seccion" | "seccion+ctx"
    "puntos de hit@5 que pierde el modelo en inglés": None, # "cero" | "pocos" | "muchos"
}

In [ ]:
def fila_barrido(nombre, tam, ov, chunks, idx):
    fila = {"estrategia": nombre, "tamaño": tam, "overlap %": ov, "chunks": len(chunks),
            "tok/chunk": sum(contar(c["texto"]) for c in chunks) // len(chunks)}
    for k in (1, 3, 5, 10):
        fila[f"hit@{k}"] = evaluar(gold, lambda q: idx.buscar(q, k=10), k=k)[f"hit@{k}"]
    fila["mrr"] = evaluar(gold, lambda q: idx.buscar(q, k=10), k=5)["mrr"]
    fila["ctx tokens @5"] = round(np.mean([sum(contar(c["texto"]) for c in idx.buscar(q["pregunta"], k=5)) for q in gold if q["evidencia"]]))
    return fila

t0 = time.time()
sec, secc = chunk_por_seccion(docs), chunk_seccion_contexto(docs)
idx_sec, idx_secc = Indice(modelo, sec), Indice(modelo, secc)
filas = [fila_barrido("seccion", "-", "-", sec, idx_sec), fila_barrido("seccion+ctx", "-", "-", secc, idx_secc)]
fijos = {}
for tam in (200, 500, 1000):
    for ov in (0, 10, 25):
        ch = chunk_fijo(docs, tam, int(tam * ov / 100), contar); fijos[(tam, ov)] = ch
        filas.append(fila_barrido("fijo", tam, ov, ch, Indice(modelo, ch)))
barrido = pd.DataFrame(filas)
print(f"{time.time()-t0:.0f} s"); print(barrido.to_markdown(index=False)); barrido

In [ ]:
# Métrica laxa frente a estricta para fijo 200 / 10 %: los mismos chunks, dos números.
f200 = fijos[(200, 10)]
for nombre, ch in (("fijo 200, laxa", f200), ("fijo 200, estricta", [{**c, "secciones": []} for c in f200]), ("seccion+ctx", secc)):
    idx = Indice(modelo, ch); r = evaluar(gold, lambda q: idx.buscar(q, k=10), k=5)
    print(f"{nombre:22s} hit@5={r['hit@5']:.2f}  mrr={r['mrr']:.2f}")

In [ ]:
# TODO: contrasta cada predicción con la tabla (acertó / falló y por qué).
CONTRASTE = {k: None for k in PREDICCIONES}

## Sección 2. Dos modelos de embeddings

In [ ]:
m_en = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("inglés: dim", m_en.get_sentence_embedding_dimension(), "| max_seq_length", m_en.max_seq_length)
comp = []
for nombre, m, msl in (("multilingüe", modelo, 512), ("inglés", m_en, 256)):
    for est, ch in (("seccion+ctx", secc), ("fijo 200", f200)):
        t = time.time(); idx = Indice(m, ch, max_seq_length=msl); dt = time.time() - t
        r = evaluar(gold, lambda q: idx.buscar(q, k=10), k=5)
        comp.append({"modelo": nombre, "dim": m.get_sentence_embedding_dimension(), "max_seq_length": msl, "chunking": est,
                     "hit@5": r["hit@5"], "mrr": r["mrr"], "embed s": round(dt, 1)})
modelo.max_seq_length = 512
print(pd.DataFrame(comp).to_markdown(index=False))

In [ ]:
# TODO: tres líneas: por qué el modelo en inglés funciona algo, por qué engaña, en qué chunking pierde menos.
EXPLICACION_MODELOS = None

## Sección 3. Gold set propio

In [ ]:
# TODO: diez preguntas sobre dos documentos no vistos en clase. Formato del gold set del curso.
# Al menos dos que crucen secciones (dos evidencias) y una sin respuesta (evidencia []).
MI_GOLD = [
    {"id": "m01", "pregunta": "Si cancelo hoy, ¿cuánto tiempo puedo seguir consultando mis facturas?", "tipo": "conceptual",
     "evidencia": [{"archivo": "07-politica-cancelacion-y-eliminacion.md", "seccion": "Qué pasa con los datos al cancelar"}]},
    {"id": "m02", "pregunta": "¿Facturio ofrece capacitación presencial?", "tipo": "sin_respuesta", "evidencia": []},
]
encabezados = {a: {s["seccion"] for s in secciones(t)} for a, t in docs.items()}
for q in MI_GOLD:
    for e in q["evidencia"]:
        ok = e["archivo"] in encabezados and e["seccion"] in encabezados[e["archivo"]]
        print(("ok       " if ok else "NO EXISTE"), q["id"], e["archivo"], "/", e["seccion"])
print(len(MI_GOLD), "preguntas;", sum(1 for q in MI_GOLD if q["evidencia"]), "con evidencia")

In [ ]:
filas_g = []
for q in MI_GOLD:
    if not q["evidencia"]:
        continue
    fila = {"id": q["id"], "pregunta": q["pregunta"][:50]}
    for nombre, idx in (("seccion+ctx", idx_secc), ("fijo 200", Indice(modelo, f200))):
        fila[f"hit@5 {nombre}"] = metricas_pregunta(idx.buscar(q["pregunta"], k=10), q["evidencia"], 5)["hit"]
    filas_g.append(fila)
if filas_g:
    print(pd.DataFrame(filas_g).to_markdown(index=False))
    for nombre, idx in (("seccion+ctx", idx_secc), ("fijo 200", Indice(modelo, f200))):
        r = evaluar(MI_GOLD, lambda q: idx.buscar(q, k=10), k=5)
        print(f"{nombre:12s} n={r['preguntas']} hit@5={r['hit@5']:.2f} mrr={r['mrr']:.2f} fallan={[x['id'] for x in r['por_pregunta'] if x['hit']==0]}")

In [ ]:
# TODO: un diagnóstico por pregunta que falle en ambas: "ambigua" | "evidencia duplicada" | "falla real", con una línea de evidencia.
DIAGNOSTICO = {}
OBSERVACION_GOLD = None   # qué aprendiste al escribir las preguntas

## Sección 4. Un filtro por metadata

In [ ]:
g = {x["id"]: x for x in gold}
r = evaluar(gold, lambda q: idx_secc.buscar(q, k=10), k=5)
print("fallan con seccion+ctx:", [x["id"] for x in r["por_pregunta"] if x["hit"] == 0])
print("tipos disponibles:", sorted(set(TIPO_DOC.values())))
# TODO: elige una pregunta que falle y un tipo de documento.
QID = "q05"
TIPO = "manual"
sin = idx_secc.buscar(g[QID]["pregunta"], k=3); con = idx_secc.buscar(g[QID]["pregunta"], k=3, filtro=lambda c: c["tipo"] == TIPO)
print(g[QID]["pregunta"]); print("evidencia :", [(e["archivo"][:2], e["seccion"]) for e in g[QID]["evidencia"]])
print("sin filtro:", [(c["archivo"][:2], c["seccion"]) for c in sin]); print(f"con {TIPO:9s}:", [(c["archivo"][:2], c["seccion"]) for c in con])

In [ ]:
# TODO: quién decide el filtro en producción y qué pasa si se equivoca.
QUIEN_DECIDE = None

## 5. Contexto con presupuesto

Con la configuración de sección más contexto jerárquico, la aplicación decide cuántos fragmentos entran al prompt (presupuesto de tokens) y en qué orden. Se mide sobre la respuesta: `q29` y `q30` con cuatro presupuestos y dos posiciones del mejor fragmento. Marca después cuáles respuestas son correctas.

In [ ]:
# TODO: predicción antes de ejecutar.
PRED_CONTEXTO = {"presupuesto_minimo": None,   # 150, 300, 600 o 900
                 "posicion_cambia": None}      # True o False

In [ ]:
def contar_tokens(texto): return int(len(texto) / 3.5) + 1

def construir_contexto(chunks, presupuesto_tokens=900, orden="mejor_primero"):
    elegidos, vistos, usados = [], set(), 0
    for c in chunks:
        clave = (c["archivo"], c["seccion"])
        if clave in vistos: continue
        t = contar_tokens(c["texto"])
        if usados + t > presupuesto_tokens: continue
        vistos.add(clave); usados += t; elegidos.append(c)
    if orden == "mejor_ultimo": elegidos = elegidos[::-1]
    partes = [f"[{i}] ({c['archivo']} · {c.get('ruta') or c['seccion']})\n{c['texto']}" for i, c in enumerate(elegidos, start=1)]
    return "\n\n".join(partes), elegidos

SISTEMA_QA = ("Eres el asistente de soporte de Facturio. Responde en español, en una o dos oraciones, "
              "únicamente con la información de los fragmentos numerados. Cita el fragmento con [n]. "
              "Si los fragmentos no contienen la respuesta, di exactamente: 'No tengo esa información en la documentación.'")

class LLM:
    """Cliente mínimo (copia de shared/llm.py): OpenAI si hay llave, modelo local si no."""
    def __init__(self, model_local="Qwen/Qwen2.5-1.5B-Instruct", model_api="gpt-5-mini"):
        self.backend = "openai" if os.environ.get("OPENAI_API_KEY") else "local"
        if self.backend == "openai":
            from openai import OpenAI
            self.client, self.model = OpenAI(), model_api
        else:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer
            self.tok = AutoTokenizer.from_pretrained(model_local)
            self.net = AutoModelForCausalLM.from_pretrained(model_local, dtype=torch.float32).eval()
            self.model = model_local
    def chat(self, user, system, max_output_tokens=160):
        t0 = time.perf_counter()
        if self.backend == "openai":
            params = dict(model=self.model, instructions=system, input=user, max_output_tokens=max_output_tokens)
            # Reasoning models (gpt-5*, o*): gastan max_output_tokens en razonar; con "minimal" la respuesta
            # cabe en presupuestos pequeños. 
            # Ejercicio: probar: 
            # "none"    Sin razonamiento adicional. Menor latencia.
            # "minimal"    Razonamiento mínimo.
            # "low"    Razonamiento ligero.
            # "medium"    Razonamiento moderado.
            # "high"    Razonamiento profundo.
            # "xhigh"    Razonamiento muy profundo.
            # Summary: la cadena de razonamiento viaja cifrada (encrypted_content) y no se puede leer; se puede
            # pedir un resumen generado por el proveedor con reasoning["summary"]:
            # "auto"    El proveedor elige el nivel de detalle.
            # "concise"    Resumen breve.
            # "detailed"    Resumen paso a paso.
            # Ejemplo: params["reasoning"] = {"effort": "medium", "summary": "auto"}
            # Ver https://developers.openai.com/api/reference/resources/responses/methods/create#responses-create-reasoning
            # https://developers.openai.com/api/docs/models/gpt-5-mini (especifico de este modelo)
            # https://developers.openai.com/api/docs/guides/reasoning?api-mode=responses
            if self.model.startswith(("gpt-5", "o1", "o3", "o4")):
                params["reasoning"] = {"effort": "minimal"}
            r = self.client.responses.create(**params)
            return {"text": r.output_text.strip(), "in": r.usage.input_tokens, "out": r.usage.output_tokens, "s": time.perf_counter() - t0}
        import torch
        texto = self.tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": user}], add_generation_prompt=True, tokenize=False)
        ids = self.tok(texto, return_tensors="pt").input_ids
        with torch.no_grad():
            out = self.net.generate(ids, max_new_tokens=max_output_tokens, do_sample=False, pad_token_id=self.tok.eos_token_id)
        nuevos = out[0, ids.shape[1]:]
        return {"text": self.tok.decode(nuevos, skip_special_tokens=True).strip(), "in": int(ids.shape[1]), "out": int(nuevos.shape[0]), "s": time.perf_counter() - t0}

def responder(llm, pregunta, retriever, presupuesto=900, orden="mejor_primero", k=8):
    chunks = retriever.buscar(pregunta, k)
    contexto, usados = construir_contexto(chunks, presupuesto, orden)
    r = llm.chat(f"<fragmentos>\n{contexto}\n</fragmentos>\n\n<pregunta>\n{pregunta}\n</pregunta>", SISTEMA_QA)
    r.update({"chunks": usados, "contexto_tokens": contar_tokens(contexto)}); return r

llm = LLM(); print("backend:", llm.backend, llm.model)
gd = {x["id"]: x for x in gold}
idx_ctx = Indice(modelo, chunk_seccion_contexto(docs))
TABLA_5 = []
for qid in ["q29", "q30"]:
    print(f"\n{qid}: {gd[qid]['pregunta']}\n  esperada: {gd[qid]['respuesta_esperada']}")
    for presupuesto in [150, 300, 600, 900]:
        for orden in ["mejor_primero", "mejor_ultimo"]:
            r = responder(llm, gd[qid]["pregunta"], idx_ctx, presupuesto=presupuesto, orden=orden)
            TABLA_5.append({"pregunta": qid, "presupuesto": presupuesto, "orden": orden, "fragmentos": len(r["chunks"]), "tokens_in": r["in"], "respuesta": r["text"]})
            print(f"  {presupuesto:4d} {orden:14s} frag={len(r['chunks'])} in={r['in']:5d} -> {r['text'][:100]}")

In [ ]:
# TODO: marca cuáles respuestas son correctas (índice de TABLA_5 -> True/False) comparando con la esperada.
CORRECTA_5 = {}   # por ejemplo {0: True, 1: True, 2: False, ...}
for i, fila in enumerate(TABLA_5):
    fila["correcta"] = CORRECTA_5.get(i)
import pandas as pd
print(pd.DataFrame(TABLA_5)[["pregunta", "presupuesto", "orden", "fragmentos", "tokens_in", "correcta"]].to_markdown(index=False))
print("\nPredicción:", PRED_CONTEXTO)

In [ ]:
# TODO: decisión. Presupuesto mínimo con respuestas correctas para ambas preguntas, efecto de la posición,
# y el presupuesto que fijarías para la mesa de soporte: tokens de entrada por consulta y al mes (1,000,000 consultas).
DECISION_5 = {"presupuesto_minimo": None, "posicion_cambio": None, "presupuesto_elegido": None, "tokens_mes": None}
print(DECISION_5)

## Sección 5. Recomendación

In [ ]:
# TODO: cada elección con la fila de la tabla que la justifica y el costo en tokens de contexto.
RECOMENDACION = {
    "presupuesto_contexto": None,   # tokens; de la sección 5
    "orden_contexto": None,         # "mejor_primero" o "mejor_ultimo"
    "chunking y tamaño": None, "modelo de embeddings": None, "k": None, "filtros": None,
    "métrica con la que se decidió": None, "tokens de contexto por consulta": None,
    "preguntas que siguen fallando": None, "por qué el chunking no las arregla": None,
}

## Resumen para el reporte

In [ ]:
print("PREDICCIONES:", json.dumps(PREDICCIONES, ensure_ascii=False, indent=1))
print("CONTRASTE:", json.dumps(CONTRASTE, ensure_ascii=False, indent=1))
print("\nBARRIDO:"); print(barrido.to_markdown(index=False))
print("\nMODELOS:"); print(pd.DataFrame(comp).to_markdown(index=False))
print("\nGOLD PROPIO:", len(MI_GOLD), "preguntas | DIAGNOSTICO:", DIAGNOSTICO)
print("FILTRO:", QID, TIPO, "| QUIEN_DECIDE:", QUIEN_DECIDE)
print("RECOMENDACION:", json.dumps(RECOMENDACION, ensure_ascii=False, indent=1))